# Generate AIForEcom Article Pages
This notebook reads the attached markdown source and generates 10 static HTML article pages using the shared AIForEcom article template.

In [ ]:
import re
from pathlib import Path
from html import escape

repo_root = Path(r'c:\Users\Ethan\Downloads\ecabnet25-star\ecabnet25-star')
md_path = Path(r'c:\Users\Ethan\Downloads\AIForEcom-Articles.md')
output_dir = repo_root

## Load and inspect the markdown source
Read the attached markdown file and verify it contains exactly 10 articles.

In [ ]:
text = md_path.read_text(encoding='utf-8')
article_blocks = re.split(r'^# Article \d+: ', text, flags=re.MULTILINE)[1:]
print(f'Found {len(article_blocks)} article blocks')
assert len(article_blocks) == 10, f'Expected 10 articles, found {len(article_blocks)}'

## Define the HTML page template
Create a reusable HTML template with placeholders for the page metadata, intro paragraph, body content, and optional CTA.

In [ ]:
base_template = '''<!DOCTYPE HTML>
<html>
  <head>
    <title>{title} — AIForEcom</title>
    <meta charset="utf-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1, user-scalable=no" />
    <meta name="description" content="{description}" />
    <link rel="stylesheet" href="assets/css/main.css" />
    <noscript><link rel="stylesheet" href="assets/css/noscript.css" /></noscript>
    <style>
      /* Nav */
      #site-nav {{ ... }}
    </style>
  </head>
  <body class="is-preload">
    <nav id="site-nav"> ... </nav>
    <section class="main style1">
      <div class="article-wrap">
        <a href="index.html" class="back-link">Back to all articles</a>
        <p class="article-meta">{meta}</p>
        <h1>{title}</h1>
        <p class="article-intro">{intro}</p>
        {body}
        {cta}
        <div class="affiliate-note">Some links in this article are affiliate links. If you sign up for a tool through a link on this page, we may earn a small commission at no extra cost to you. We only recommend tools we've actually tested.</div>
      </div>
    </section>
    <section id="footer"> ... </section>
    <script src="assets/js/jquery.min.js"></script>
  </body>
</html>'''

## Map filenames, metadata, and titles
Create a mapping of the 10 output filenames to their article titles, category metadata, and any additional CTA block requirements.

In [ ]:
mapping = {
    'best-ai-tools-shopify.html': ('Hub guide', 'Best AI Tools for Shopify Stores (2026)', ''),
    'ai-product-description-generators.html': ('Comparison', 'AI Product Description Generators: Honest Review (2026)', ''),
    'ai-amazon-listing-optimisation.html': ('Tutorial', 'How to Use AI to Write Amazon Listing Titles and Bullet Points', ''),
    'tidio-vs-gorgias.html': ('Comparison', 'Tidio vs Gorgias — Which AI Customer Support Tool Is Right for Your Store?', '<ul class="actions" style="margin-top:2em">\n  <li><a href="https://www.tidio.com/?ref=aiforecom" class="button primary" target="_blank" rel="noopener">Try Tidio free</a></li>\n</ul>'),
    'jasper-ai-review-ecommerce.html': ('In-depth review', 'Jasper AI Review for E-Commerce — Is It Worth It for Shopify Sellers?', ''),
    'jasper-vs-copyai-ecommerce.html': ('Comparison', 'Jasper vs Copy.ai for E-Commerce — Which Is Better in 2026?', ''),
    'best-ai-tools-etsy-sellers.html': ('Platform guide', 'Best AI Tools for Etsy Sellers (2026)', ''),
    'ai-ecommerce-email-marketing.html': ('Tutorial', 'How to Use AI for E-Commerce Email Marketing (Omnisend vs Klaviyo)', '<ul class="actions" style="margin-top:2em">\n  <li><a href="https://www.omnisend.com/?ref=aiforecom" class="button primary" target="_blank" rel="noopener">Try Omnisend free</a></li>\n</ul>'),
    'ai-product-photo-tools.html': ('Listicle', 'AI Product Photo Tools for Shopify', ''),
    'shopify-ai-tool-stack.html': ('Stack guide', 'The AI Tool Stack Every Shopify Store Should Have in 2026 (Under $100/Month)', '<div style="background:#f5f5ff;border:1px solid #c7d2fe;border-radius:6px;padding:1.4em 1.6em;margin:2em 0;">\n  <strong>Not sure which stack suits your store?</strong>\n  <p style="margin:0.5em 0 1em">Take our free AI Stack Finder quiz — five questions, personalised recommendation, no email required.</p>\n  <ul class="actions">\n    <li><a href="stack-finder.html" class="button primary">Take the AI Stack Finder</a></li>\n  </ul>\n</div>'),
}

## Convert markdown content to HTML
Parse each article's markdown content, convert headings, paragraphs, lists, tables, and bold text into proper HTML, and preserve only the first paragraph as the article intro.

In [ ]:
def escape_html(text: str) -> str:
    return escape(text, quote=False)

def inline_format(text: str) -> str:
    text = escape_html(text)
    text = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', text)
    text = re.sub(r'__(.+?)__', r'<strong>\1</strong>', text)
    text = re.sub(r'`([^`]+)`', r'<code>\1</code>', text)
    return text

def render_table(lines):
    rows = [[cell.strip() for cell in line.strip().strip('|').split('|')] for line in lines]
    html = ['<table>']
    if len(rows) >= 2 and re.match(r'^[-:]+$', rows[1][0]):
        header = rows[0]
        html.append('<thead><tr>' + ''.join(f'<th>{inline_format(cell)}</th>' for cell in header) + '</tr></thead>')
        body_rows = rows[2:]
    elif len(rows) >= 2 and all(re.match(r'^[-:]+$', cell) for cell in rows[1]):
        header = rows[0]
        html.append('<thead><tr>' + ''.join(f'<th>{inline_format(cell)}</th>' for cell in header) + '</tr></thead>')
        body_rows = rows[2:]
    else:
        body_rows = rows
    if body_rows:
        html.append('<tbody>')
        for row in body_rows:
            html.append('<tr>' + ''.join(f'<td>{inline_format(cell)}</td>' for cell in row) + '</tr>')
        html.append('</tbody>')
    html.append('</table>')
    return html

def convert_markdown_to_html(md: str) -> str:
    lines = md.splitlines()
    html_lines = []
    in_ul = False
    in_ol = False
    in_code = False
    table_lines = []

    def close_lists():
        nonlocal in_ul, in_ol
        if in_ul:
            html_lines.append('</ul>')
            in_ul = False
        if in_ol:
            html_lines.append('</ol>')
            in_ol = False

    for line in lines:
        if line.startswith('```'):
            if in_code:
                html_lines.append('</code></pre>')
                in_code = False
            else:
                close_lists()
                if table_lines:
                    html_lines.extend(render_table(table_lines))
                    table_lines = []
                html_lines.append('<pre><code>')
                in_code = True
            continue
        if in_code:
            html_lines.append(escape_html(line))
            continue
        if line.strip() == '' or line.strip() == '---':
            close_lists()
            if table_lines:
                html_lines.extend(render_table(table_lines))
                table_lines = []
            continue
        if line.startswith('### '):
            close_lists()
            if table_lines:
                html_lines.extend(render_table(table_lines))
                table_lines = []
            html_lines.append(f'<h3>{inline_format(line[4:])}</h3>')
            continue
        if line.startswith('## '):
            close_lists()
            if table_lines:
                html_lines.extend(render_table(table_lines))
                table_lines = []
            html_lines.append(f'<h2>{inline_format(line[3:])}</h2>')
            continue
        if line.startswith('|'):
            table_lines.append(line)
            continue
        ordered_match = re.match(r'^(\d+)\.\s+(.*)', line)
        if ordered_match:
            close_lists()
            if not in_ol:
                html_lines.append('<ol>')
                in_ol = True
            html_lines.append(f'<li>{inline_format(ordered_match.group(2))}</li>')
            continue
        if line.startswith('- ') or line.startswith('* '):
            close_lists()
            if not in_ul:
                html_lines.append('<ul>')
                in_ul = True
            html_lines.append(f'<li>{inline_format(line[2:])}</li>')
            continue
        if table_lines:
            html_lines.extend(render_table(table_lines))
            table_lines = []
        close_lists()
        html_lines.append(f'<p>{inline_format(line)}</p>')

    close_lists()
    if in_code:
        html_lines.append('</code></pre>')
    if table_lines:
        html_lines.extend(render_table(table_lines))
    return '
'.join(html_lines)

def extract_first_paragraph(html: str) -> str:
    m = re.search(r'<p>(.*?)</p>', html, re.DOTALL)
    return m.group(1) if m else ''

## Generate each article page and write files
Render each article page with the converted HTML content and write the resulting files.

In [ ]:
created_files = []
for idx, block in enumerate(article_blocks):
    lines = block.splitlines()
    article_title = lines[0].strip()
    raw_body = '\n'.join(lines[1:]).strip()

    filename = list(mapping.keys())[idx]
    meta, expected_title, cta = mapping[filename]
    if article_title != expected_title:
        print(f'Warning: title mismatch for {filename}: 
 != 
')

    body_html = convert_markdown_to_html(raw_body)
    intro = extract_first_paragraph(body_html)
    if not intro:
        intro = expected_title
    description = intro.replace('"', '&quot;')[:160]

    page = base_template.format(
        title=expected_title,
        description=description,
        meta=meta,
        intro=intro,
        body=body_html,
        cta=cta,
    )
    out_path = output_dir / filename
    out_path.write_text(page, encoding='utf-8')
    created_files.append(filename)

print('Created files:')
for name in created_files:
    print('-', name)

## Verify created filenames
Confirm that all 10 generated HTML files exist in the repository.

In [ ]:
for filename in mapping:
    print(filename, 'FOUND' if (output_dir / filename).exists() else 'MISSING')